# Predicting Electric Vehicle Purchases

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier




In [2]:
train_df = pd.read_csv("C:/Users/deepz/Downloads/python_projects/ev_purchase/train.csv")
test_df = pd.read_csv("C:/Users/deepz/Downloads/python_projects/ev_purchase/test.csv")

print("train shape:",train_df.shape)
print("test shape:",test_df.shape)

print("train col names:", train_df.columns)
print("test col names:", test_df.columns)

print("train data types:", train_df.dtypes)
print("test data types:", test_df.dtypes)

print("train missing vals:", train_df.isna().sum().sum())
print("test missing vals:", test_df.isna().sum().sum())

print("train city_type unique vals:", train_df['City_Type'].unique())
print("train car_type unique vals:", train_df['Current_Car_Type'].unique())

print("train df will buy distribution:", train_df["Will_Buy_EV"].value_counts(normalize=True) * 100)


train shape: (668665, 15)
test shape: (286571, 14)
train col names: Index(['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender',
       'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV'],
      dtype='str')
test col names: Index(['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender',
       'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='str')
train data types: id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_S

In [3]:
X = train_df.iloc[:,:-1]
X["hidden_score"] = (
    1.2 * (X["Annual_Income_USD"] / 100000)
    + 0.6 * X["Environmental_Concern_Level"]
    + 2 * (X["Subsidy_Available"] == "Yes").astype(int)
    - 1 * (X["Range_Anxiety_Level"] == "Medium").astype(int)
    - 3 * (X["Range_Anxiety_Level"] == "High").astype(int)
)

y = train_df["Will_Buy_EV"].map({"No": 0, "Yes": 1})
x_ids = X['id']
X = X.drop(columns='id')
#print(X.head)

In [4]:
X['Home_Charging_Possible'] = X['Home_Charging_Possible'].map({'Yes': 1, "No": 0})
X['Subsidy_Available'] = X['Subsidy_Available'].map({'Yes': 1, "No": 0})
#X['Range_Anxiety_Level'] = X['Range_Anxiety_Level'].map({'Low':1,"Medium":2,"High":3})
# X['Annual_Income_USD_squared'] = X['Annual_Income_USD'] ** 2
# X['Daily_Commute_km_squared'] = X['Daily_Commute_km'] ** 2

city_type_encoded = pd.get_dummies(X["City_Type"], dtype=int)
Current_Car_Type_encoded = pd.get_dummies(X["Current_Car_Type"],dtype=int)
gender_encoded = pd.get_dummies(X["Gender"], dtype=int)
Environmental_Concern_Level_encoded = pd.get_dummies(X["Environmental_Concern_Level"], dtype=int, prefix="EnvConcern")
Range_Anxiety_Level_encoded = pd.get_dummies(X['Range_Anxiety_Level'], dtype=int, prefix='AnxLev')


X = pd.concat([X,city_type_encoded],axis=1)
X = pd.concat([X,Current_Car_Type_encoded],axis=1)
X = pd.concat([X,gender_encoded],axis=1)
X = pd.concat([X,Environmental_Concern_Level_encoded], axis=1)
X = pd.concat([X,Range_Anxiety_Level_encoded], axis=1)

X = X.drop(columns=['City_Type', 'Current_Car_Type', 'Gender', 'Range_Anxiety_Level', 'Environmental_Concern_Level'])
print(X.columns)


Index(['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
       'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
       'Home_Charging_Possible', 'Subsidy_Available', 'hidden_score', 'Rural',
       'Suburban', 'Urban', 'Hatchback', 'SUV', 'Sedan', 'Truck', 'Female',
       'Male', 'Other', 'EnvConcern_1.0', 'EnvConcern_2.0', 'EnvConcern_3.0',
       'EnvConcern_4.0', 'EnvConcern_5.0', 'AnxLev_High', 'AnxLev_Low',
       'AnxLev_Medium'],
      dtype='str')


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

numerical_columns = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work'
]


scaler = StandardScaler()


scaler.fit(X_train[numerical_columns])


X_train[numerical_columns] = scaler.transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [6]:
model = LogisticRegression(class_weight=None, random_state=42, solver='lbfgs', max_iter=5000, C=1.0)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression ROC AUC:", roc_auc)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Logistic Regression ROC AUC: 0.9383707141351446
Accuracy: 0.8946109038158121
              precision    recall  f1-score   support

           0       0.93      0.94      0.94    110377
           1       0.71      0.66      0.69     23356

    accuracy                           0.89    133733
   macro avg       0.82      0.80      0.81    133733
weighted avg       0.89      0.89      0.89    133733

